## Deliverable 2

task 1:

In [ ]:
from pathlib import Path 

import re

def read_file(file_path):

    path = Path(file_path)

    try: 
        file_text_content = path.read_text(encoding="utf-8")
        return(file_text_content, 'utf-8')

    except UnicodeDecodeError:
        file_text_content = path.read_text(encoding="iso-8859-1")
        return(file_text_content, 'iso-8859-1') 

def parse_log(log_text):
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})] "
        r"\[(?P<register_id>[^\]]+)\] "
        r"item=(?P<item>\w+) "
        r"price=(?P<price>\d+\.\d+) "
        r"(?P<currency>[A-Z]{3}) "
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
        )
    records = []
    for line in log_text.splitlines():
        match = pattern.search(line)

        if match:
            record = match.groupdict()

            record["price"] = float(record["price"])

            records.append(record)
    return records 


Task 2: 


In [ ]:
import json

def find_log_files(dir_path): 
    dir_path = Path(dir_path)
    log_files = []


    for item in dir_path.iterdir():
        if item.is_dir(): 
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log": 
            log_files.append(item)
    return log_files

def find_active_registers(data):
    active_registers = set()

    if isinstance(data, dict):
        if "id" in data and data.get("active") is True:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))


    elif isinstance(data, list):

        for item in data: 
            active_registers.update(find_active_registers(item))

    return active_registers

with open("data/store_config.json", "r", encoding="utf-8") as f: 
    metadata = json.load(f)

active_registers_in_metadata = find_active_registers(metadata)


Task 3: 

In [3]:
import numpy as np


def removing_price_outliers(data): 
    
    price = np.array(data, dtype=float).ravel()
    mean = np.mean(price)
    std = np.std(price)

    outliers = np.abs(price - mean) > 3 * std

    valid_prices = price[~outliers]
    valid_median = np.median(valid_prices)

    price[outliers] = valid_median 

    return price


def compute_refund_ratio(sales_count: np.ndarray,refund_count: np.ndarray) -> np.ndarray:

    refund_ratio = refund_count/(sales_count + 10**-6) 
    
    return refund_ratio


Task 4: 

In [12]:
import pandas


def convert_transaction_dicts_df(records):
    df = pandas.DataFrame(records)

    if "timestamp" in df.columns:
        df['timestamp'] = pandas.to_datetime(df["timestamp"])

        df = df.sort_values(by="timestamp", ascending=True, ignore_index=True)

    return df 

def total_price_per_hour(df): 

    per_hour = df.groupby(["item", pandas.Grouper(key="timestamp", freq="1h")])["price"].sum().reset_index()

    return per_hour

def create_pivot_table(df): 
    pivot_table = df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )
    return pivot_table

def refund_per_item(df):
    counts = pandas.crosstab(df["item"], df["status"])

    sales = counts["SUCCESS"].to_numpy()
    refunds = counts["REFUND"].to_numpy()

    ratios = compute_refund_ratio(sales,refunds)

    return pandas.Series(ratios, index=counts.index, name="refund_ratio")
    

Task 5: 

In [ ]:

def workflow():
    def extracting_active_registers(data_path=Path("data")):
    
        log_files = find_log_files(data_path)

        json_path = data_path / "store_config.json"
        with open(json_path, "r", encoding="utf-8") as f: 
            metadata = json.load(f)

        active_registers = find_active_registers(metadata)


        return active_registers, log_files


    def ingest_txtlogs_parse_records(log_files):
        all_records = []

        for file_path in log_files: 
            content, encoding = read_file(file_path)
            records = parse_log(content)

            all_records.extend(records)

        return all_records
    
    def create_initial_dataframe(log_files):
        records = ingest_txtlogs_parse_records(log_files)

        initial_df = convert_transaction_dicts_df(records)

        return(initial_df)

   




    



    

    





=== TEST AV INGEST-FUNKSJONEN ===
Totalt antall hentede poster: 720
Datatype på resultat:        <class 'list'>
Datatype på første element:  <class 'dict'>

De 2 første transaksjonene:
{'timestamp': '2026-10-15 07:01:36', 'register_id': 'POS_01', 'item': 'Muffin', 'price': 3.6, 'currency': 'USD', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 07:01:57', 'register_id': 'POS_01', 'item': 'Muffin', 'price': 3.21, 'currency': 'NOK', 'status': 'SUCCESS'}
